In [5]:
from pathlib import Path
import hashlib
import pandas as pd
import mne

# Toggle this to trade off speed vs. exact duplicate detection.
COMPUTE_HASHES = True

mne.set_log_level("ERROR")

root = Path.cwd()
raw_dir_candidates = [root / "data" / "raw", root.parent / "data" / "raw"]
raw_dir = next((p for p in raw_dir_candidates if p.exists()), None)
if raw_dir is None:
    raise FileNotFoundError(
        "Could not find data/raw. Checked: " + ", ".join(str(p) for p in raw_dir_candidates)
    )

gdf_files = sorted(raw_dir.glob("*.gdf"))
print(f"GDF files found: {len(gdf_files)}")


def file_hash(path: Path, algo: str = "sha256", chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.new(algo)
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


rows = []
for path in gdf_files:
    size_mb = path.stat().st_size / (1024 * 1024)
    sha256 = file_hash(path) if COMPUTE_HASHES else ""

    try:
        raw = mne.io.read_raw_gdf(path, preload=False, verbose="error")
        n_channels = len(raw.ch_names)
        sfreq = float(raw.info["sfreq"])
        n_times = int(raw.n_times)
        duration_sec = n_times / sfreq if sfreq > 0 else float("nan")
        ch_types = ",".join(sorted(set(raw.get_channel_types())))

        annotations = raw.annotations
        n_ann = len(annotations)
        ann_desc = ",".join(sorted(set([str(d) for d in annotations.description]))) if n_ann else ""

        event_count = 0
        event_desc = ""
        if n_ann:
            events, event_id = mne.events_from_annotations(raw)
            event_count = len(events)
            event_desc = ",".join(sorted([str(k) for k in event_id.keys()]))

        rows.append(
            {
                "file": path.name,
                "size_mb": round(size_mb, 2),
                "sha256": sha256,
                "n_channels": n_channels,
                "sfreq": sfreq,
                "n_times": n_times,
                "duration_sec": round(duration_sec, 2),
                "ch_types": ch_types,
                "n_annotations": n_ann,
                "annotation_descriptions": ann_desc,
                "n_events_from_annotations": event_count,
                "event_descriptions": event_desc,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "file": path.name,
                "size_mb": round(size_mb, 2),
                "sha256": sha256,
                "error": repr(exc),
            }
        )


df = pd.DataFrame(rows).sort_values("file")

# Summary stats
if not df.empty:
    if "sfreq" in df.columns:
        unique_sfreq = sorted(df["sfreq"].dropna().unique().tolist())
        print("Unique sfreq values:", unique_sfreq)

    if "n_channels" in df.columns:
        unique_ch = sorted(df["n_channels"].dropna().unique().tolist())
        print("Unique channel counts:", unique_ch)

    if "duration_sec" in df.columns:
        print(
            "Duration sec (min/median/max):",
            float(df["duration_sec"].min()),
            float(df["duration_sec"].median()),
            float(df["duration_sec"].max()),
        )

    if "n_annotations" in df.columns:
        print("Files with annotations:", int((df["n_annotations"] > 0).sum()))

# Duplicate checks
size_counts = df["size_mb"].value_counts() if "size_mb" in df.columns else pd.Series()
size_dups = size_counts[size_counts > 1]
if not size_dups.empty:
    print("\nPotential duplicates by size (not exact):")
    for size_mb, count in size_dups.items():
        files = df[df["size_mb"] == size_mb]["file"].tolist()
        print(f"- {size_mb} MB -> {files}")

if COMPUTE_HASHES and "sha256" in df.columns:
    hash_counts = df["sha256"].value_counts()
    dup_hashes = hash_counts[hash_counts > 1].index.tolist()
    if dup_hashes:
        print("\nExact duplicates by sha256:")
        for h in dup_hashes:
            files = df[df["sha256"] == h]["file"].tolist()
            print(f"- {h[:12]}... -> {files}")
    else:
        print("\nNo exact duplicates by sha256.")

print("\nPreview:")
print(df.head(10).to_string(index=False))

# Keep df in memory for further analysis


GDF files found: 37
Unique sfreq values: [250.0]
Unique channel counts: [24]
Duration sec (min/median/max): 599.81 605.82 605.95
Files with annotations: 37

Potential duplicates by size (not exact):
- 27.74 MB -> ['ID1.gdf', 'ID10.gdf', 'ID11.gdf', 'ID13.gdf', 'ID14.gdf', 'ID15(1).gdf', 'ID15.gdf', 'ID16.gdf', 'ID18.gdf', 'ID19.gdf', 'ID2.gdf', 'ID20.gdf', 'ID21.gdf', 'ID22.gdf', 'ID23.gdf', 'ID24.gdf', 'ID25.gdf', 'ID27.gdf', 'ID3.gdf', 'ID30.gdf', 'ID31.gdf', 'ID33.gdf', 'ID35.gdf', 'ID37.gdf', 'ID38.gdf', 'ID39.gdf', 'ID4.gdf', 'ID40.gdf', 'ID41.gdf', 'ID43.gdf', 'ID5.gdf', 'ID6.gdf', 'ID7.gdf', 'ID8.gdf', 'ID9.gdf']
- 27.46 MB -> ['ID0.gdf', 'ID26.gdf']

Exact duplicates by sha256:
- ecc0aa3b9170... -> ['ID15(1).gdf', 'ID15.gdf']

Preview:
       file  size_mb                                                           sha256  n_channels  sfreq  n_times  duration_sec ch_types  n_annotations annotation_descriptions  n_events_from_annotations event_descriptions
    ID0.gdf    27.46 a7a

In [6]:
from pathlib import Path
import hashlib
import numpy as np
import mne

mne.set_log_level("ERROR")


def resolve_raw_dir() -> Path:
    root = Path.cwd()
    candidates = [root / "data" / "raw", root.parent / "data" / "raw"]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find data/raw. Checked: " + ", ".join(str(p) for p in candidates)
    )


raw_dir = raw_dir if "raw_dir" in globals() and raw_dir.exists() else resolve_raw_dir()

id15_path = raw_dir / "ID15.gdf"
id15_copy_path = raw_dir / "ID15(1).gdf"

if not id15_path.exists() or not id15_copy_path.exists():
    raise FileNotFoundError(
        f"Missing file(s): {id15_path.name} or {id15_copy_path.name}"
    )


def file_hash_local(path: Path, algo: str = "sha256", chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.new(algo)
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


size_a = id15_path.stat().st_size
size_b = id15_copy_path.stat().st_size
sha_a = file_hash_local(id15_path)
sha_b = file_hash_local(id15_copy_path)

print(f"Size equal: {size_a == size_b} ({size_a} vs {size_b} bytes)")
print(f"SHA256 equal: {sha_a == sha_b}")

raw_a = mne.io.read_raw_gdf(id15_path, preload=True, verbose="error")
raw_b = mne.io.read_raw_gdf(id15_copy_path, preload=True, verbose="error")

same_sfreq = raw_a.info["sfreq"] == raw_b.info["sfreq"]
same_channels = raw_a.ch_names == raw_b.ch_names
same_n_times = raw_a.n_times == raw_b.n_times

data_a = raw_a.get_data()
data_b = raw_b.get_data()
data_equal = np.array_equal(data_a, data_b)
max_abs_diff = float(np.max(np.abs(data_a - data_b))) if not data_equal else 0.0

ann_a = raw_a.annotations
ann_b = raw_b.annotations
ann_equal = (
    len(ann_a) == len(ann_b)
    and np.allclose(ann_a.onset, ann_b.onset)
    and np.allclose(ann_a.duration, ann_b.duration)
    and list(ann_a.description) == list(ann_b.description)
)

print(f"Same sfreq: {same_sfreq}")
print(f"Same channels: {same_channels}")
print(f"Same n_times: {same_n_times}")
print(f"Data array equal: {data_equal}")
print(f"Max abs diff: {max_abs_diff}")
print(f"Annotations equal: {ann_equal}")


Size equal: True (29085978 vs 29085978 bytes)
SHA256 equal: True
Same sfreq: True
Same channels: True
Same n_times: True
Data array equal: True
Max abs diff: 0.0
Annotations equal: True


In [7]:
from pathlib import Path
import re
import pandas as pd
import mne

mne.set_log_level("ERROR")


def resolve_raw_dir() -> Path:
    root = Path.cwd()
    candidates = [root / "data" / "raw", root.parent / "data" / "raw"]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find data/raw. Checked: " + ", ".join(str(p) for p in candidates)
    )


raw_dir = raw_dir if "raw_dir" in globals() and raw_dir.exists() else resolve_raw_dir()

gdf_files = sorted(raw_dir.glob("*.gdf"))

EXPECTED_TOTAL_SEC = 10 * 60
EXPECTED_HALF_SEC = 5 * 60
TOL_SEC = 30

open_tokens = ["eyes open", "open eyes", "open", "eo", "ojos abiertos", "abiertos"]
closed_tokens = ["eyes closed", "closed eyes", "closed", "ec", "ojos cerrados", "cerrados"]


def normalize_desc(desc: str) -> str:
    return re.sub(r"\s+", " ", str(desc).strip().lower())


def label_from_desc(desc: str) -> str | None:
    d = normalize_desc(desc)
    if any(tok in d for tok in open_tokens):
        return "open"
    if any(tok in d for tok in closed_tokens):
        return "closed"
    return None


rows = []
for path in gdf_files:
    row = {
        "file": path.name,
        "total_sec": float("nan"),
        "open_sec": float("nan"),
        "closed_sec": float("nan"),
        "open_start": float("nan"),
        "closed_start": float("nan"),
        "markers_found": 0,
        "total_ok": False,
        "open_ok": False,
        "closed_ok": False,
        "open_start_ok": False,
        "closed_start_ok": False,
        "method": "no_markers",
        "error": "",
    }

    try:
        raw = mne.io.read_raw_gdf(path, preload=False, verbose="error")
        sfreq = float(raw.info["sfreq"])
        total_sec = raw.n_times / sfreq if sfreq > 0 else float("nan")
        row["total_sec"] = round(total_sec, 2)

        ann_rows = []
        for desc, onset, duration in zip(
            raw.annotations.description, raw.annotations.onset, raw.annotations.duration
        ):
            label = label_from_desc(desc)
            if label:
                ann_rows.append(
                    {
                        "label": label,
                        "onset": float(onset),
                        "duration": float(duration),
                        "desc": str(desc),
                    }
                )
        row["markers_found"] = len(ann_rows)

        open_sec = float("nan")
        closed_sec = float("nan")
        open_start = float("nan")
        closed_start = float("nan")
        method = "no_markers"

        if ann_rows:
            ann_df = pd.DataFrame(ann_rows).sort_values("onset")
            open_onsets = ann_df.loc[ann_df["label"] == "open", "onset"].tolist()
            closed_onsets = ann_df.loc[ann_df["label"] == "closed", "onset"].tolist()

            open_start = min(open_onsets) if open_onsets else float("nan")
            closed_start = min(closed_onsets) if closed_onsets else float("nan")

            if (ann_df["duration"] > 0).any():
                method = "duration"
                open_sec = ann_df.loc[ann_df["label"] == "open", "duration"].sum()
                closed_sec = ann_df.loc[ann_df["label"] == "closed", "duration"].sum()
            elif open_onsets and closed_onsets:
                method = "onset"
                if open_start < closed_start:
                    open_sec = closed_start - open_start
                    closed_sec = total_sec - closed_start
                else:
                    closed_sec = open_start - closed_start
                    open_sec = total_sec - open_start
            else:
                method = "markers_missing"

        row["open_sec"] = round(open_sec, 2) if open_sec == open_sec else float("nan")
        row["closed_sec"] = round(closed_sec, 2) if closed_sec == closed_sec else float("nan")
        row["open_start"] = round(open_start, 2) if open_start == open_start else float("nan")
        row["closed_start"] = (
            round(closed_start, 2) if closed_start == closed_start else float("nan")
        )
        row["method"] = method

        row["total_ok"] = (
            abs(total_sec - EXPECTED_TOTAL_SEC) <= TOL_SEC if total_sec == total_sec else False
        )
        row["open_ok"] = (
            abs(open_sec - EXPECTED_HALF_SEC) <= TOL_SEC if open_sec == open_sec else False
        )
        row["closed_ok"] = (
            abs(closed_sec - EXPECTED_HALF_SEC) <= TOL_SEC if closed_sec == closed_sec else False
        )
        row["open_start_ok"] = (
            abs(open_start - 0.0) <= TOL_SEC if open_start == open_start else False
        )
        row["closed_start_ok"] = (
            abs(closed_start - EXPECTED_HALF_SEC) <= TOL_SEC
            if closed_start == closed_start
            else False
        )
    except Exception as exc:
        row["error"] = repr(exc)

    rows.append(row)


df_open_closed = pd.DataFrame(rows).sort_values("file")
print(df_open_closed.head(10).to_string(index=False))

if not df_open_closed.empty:
    print("\nSummary:")
    print("Files with open/closed markers:", int((df_open_closed["markers_found"] > 0).sum()))
    print("Total duration ~10 min:", int(df_open_closed["total_ok"].sum()))
    print("Open segment ~5 min:", int(df_open_closed["open_ok"].sum()))
    print("Closed segment ~5 min:", int(df_open_closed["closed_ok"].sum()))
    print("Open starts near 0 sec:", int(df_open_closed["open_start_ok"].sum()))
    print("Closed starts near 5 min:", int(df_open_closed["closed_start_ok"].sum()))

    failures = df_open_closed[
        (df_open_closed["markers_found"] > 0)
        & (
            ~df_open_closed["open_ok"]
            | ~df_open_closed["closed_ok"]
            | ~df_open_closed["open_start_ok"]
            | ~df_open_closed["closed_start_ok"]
        )
    ]
    if not failures.empty:
        print("\nFiles with open/closed markers but unexpected timing:")
        print(
            failures[
                [
                    "file",
                    "total_sec",
                    "open_sec",
                    "closed_sec",
                    "open_start",
                    "closed_start",
                    "method",
                ]
            ].to_string(index=False)
        )

    errors = df_open_closed[df_open_closed["error"] != ""]
    if not errors.empty:
        print("\nFiles with read errors:")
        print(errors[["file", "error"]].to_string(index=False))


       file  total_sec  open_sec  closed_sec  open_start  closed_start  markers_found  total_ok  open_ok  closed_ok  open_start_ok  closed_start_ok     method error
    ID0.gdf     599.81       NaN         NaN         NaN           NaN              0      True    False      False          False            False no_markers      
    ID1.gdf     605.82       NaN         NaN         NaN           NaN              0      True    False      False          False            False no_markers      
   ID10.gdf     605.95       NaN         NaN         NaN           NaN              0      True    False      False          False            False no_markers      
   ID11.gdf     605.82       NaN         NaN         NaN           NaN              0      True    False      False          False            False no_markers      
   ID13.gdf     605.82       NaN         NaN         NaN           NaN              0      True    False      False          False            False no_markers      
   ID14.gd

In [10]:
from pathlib import Path
import re
import pandas as pd
import yaml


def find_project_root() -> Path:
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent]
    for p in candidates:
        if (p / "configs" / "preprocessing.yaml").exists():
            return p
    raise FileNotFoundError(
        "Could not find configs/preprocessing.yaml in cwd or parent."
    )


root = find_project_root()

cfg_path = root / "configs" / "preprocessing.yaml"
with cfg_path.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

questionnaire_rel = cfg["questionnaire"]["raw_data_path"]
questionnaire_path = root / questionnaire_rel
if not questionnaire_path.exists():
    raise FileNotFoundError(f"Missing questionnaire file: {questionnaire_path}")

raw_dir = raw_dir if "raw_dir" in globals() and raw_dir.exists() else root / "data" / "raw"

df_q = pd.read_excel(questionnaire_path)

rows_to_drop = cfg["questionnaire"].get("rows_to_drop", [])
if rows_to_drop:
    df_q = df_q.drop(index=rows_to_drop, errors="ignore")

id_col = "ID"
pain_col = cfg["questionnaire"]["pain_scale"]["source_column"]
bins = cfg["questionnaire"]["pain_scale"]["bins"]
labels = cfg["questionnaire"]["pain_scale"]["labels"]

id_series = pd.to_numeric(df_q[id_col], errors="coerce")
pain_scores = pd.to_numeric(df_q[pain_col], errors="coerce")

pain_label = pd.cut(pain_scores, bins=bins, labels=labels, include_lowest=True)

codes = pd.Categorical(pain_label, categories=labels, ordered=True).codes
codes = pd.Series(codes).replace({-1: pd.NA}).astype("Int64")

df_q["pain_scale_label"] = pain_label.astype("string")
df_q["pain_scale_code"] = codes

df_q_clean = df_q.dropna(subset=[id_col]).copy()
df_q_clean[id_col] = pd.to_numeric(df_q_clean[id_col], errors="coerce").astype("Int64")


gdf_files = sorted(raw_dir.glob("*.gdf"))
rows = []
for path in gdf_files:
    match = re.search(r"ID(\d+)", path.stem, flags=re.IGNORECASE)
    rows.append(
        {
            "file": path.name,
            "id": int(match.group(1)) if match else pd.NA,
        }
    )

df_gdf = pd.DataFrame(rows)

merged = df_gdf.merge(
    df_q_clean[[id_col, "pain_scale_label", "pain_scale_code"]],
    left_on="id",
    right_on=id_col,
    how="left",
)

label_to_code = {label: i for i, label in enumerate(labels)}
print("Pain scale mapping (label -> code):", label_to_code)

print("GDF files:", len(df_gdf))
print("GDF files with pain scale:", int(merged["pain_scale_code"].notna().sum()))
print("GDF files missing pain scale:", int(merged["pain_scale_code"].isna().sum()))

print("\nExample mapping:")
print(merged.head(36).to_string(index=False))


dup_gdf = merged["id"].value_counts(dropna=True)
dup_gdf = dup_gdf[dup_gdf > 1]
if not dup_gdf.empty:
    print("\nDuplicate IDs in GDF files:")
    for id_val, _ in dup_gdf.items():
        files = merged[merged["id"] == id_val]["file"].tolist()
        print(f"- ID{id_val}: {files}")


dup_q = df_q_clean[id_col].value_counts()
dup_q = dup_q[dup_q > 1]
if not dup_q.empty:
    print("\nDuplicate IDs in questionnaire:")
    for id_val, count in dup_q.items():
        print(f"- ID{id_val}: {count} rows")


Pain scale mapping (label -> code): {'Low': 0, 'Moderate': 1, 'Severe': 2}
GDF files: 36
GDF files with pain scale: 36
GDF files missing pain scale: 0

Example mapping:
    file  id  ID pain_scale_label  pain_scale_code
 ID0.gdf   0   0           Severe                2
 ID1.gdf   1   1         Moderate                1
ID10.gdf  10  10              Low                0
ID11.gdf  11  11         Moderate                1
ID13.gdf  13  13              Low                0
ID14.gdf  14  14              Low                0
ID15.gdf  15  15           Severe                2
ID16.gdf  16  16         Moderate                1
ID18.gdf  18  18         Moderate                1
ID19.gdf  19  19           Severe                2
 ID2.gdf   2   2              Low                0
ID20.gdf  20  20           Severe                2
ID21.gdf  21  21         Moderate                1
ID22.gdf  22  22           Severe                2
ID23.gdf  23  23         Moderate                1
ID24.gdf  24  2